In [26]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix
)

# --- HYPERPARAMETERS ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {DEVICE}")

IMAGE_SIZE = 380 
BATCH_SIZE = 8       
EPOCHS = 4           
LR = 1e-4
VAL_SPLIT = 0.1      # 10% of training data will be used for validation

# --- UPDATED DATA PATHS ---
TRAIN_SHARD_PATHS = [
    "/kaggle/input/datasets/zeus005/shard-1/shard_1",
    "/kaggle/input/datasets/zeus005/shard-2/shard_2"
]

# This folder is for INFERENCE ONLY (no labels)
TEST_IMAGES_PATH = "/kaggle/input/datasets/zeus005/validation-set-ivp/val_images"

# --- CUSTOM TRANSFORMS ---
class GaussianNoise(object):
    def __init__(self, mean=0., std=1.):
        self.std = std
        self.mean = mean
        
    def __call__(self, tensor):
        return tensor + torch.randn(tensor.size()) * self.std + self.mean

# --- ROBUST AUGMENTATION PIPELINE (TRAIN) ---
TRAIN_TRANSFORMS = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.2, 0.1)], p=0.3),
    transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 2.0))], p=0.3),
    transforms.RandomApply([GaussianNoise(0, 0.05)], p=0.3),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- CLEAN PIPELINE (VAL/TEST) ---
VAL_TRANSFORMS = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

Using Device: cuda


In [19]:
class AIGenDetDataset(Dataset):
    def __init__(self, shard_paths, transform=None):
        self.transform = transform
        self.shard_dirs = [x for x in shard_paths if os.path.exists(x)]
        
        if not self.shard_dirs:
            raise RuntimeError(f"No valid shard directories found in {shard_paths}")

        label_dfs = []
        for s_dir in self.shard_dirs:
            # Check for labels.csv in each shard folder
            csv_path = os.path.join(s_dir, 'labels.csv')
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path, index_col=0)
                df['shard_path'] = s_dir 
                label_dfs.append(df)
            else:
                print(f"Warning: labels.csv not found in {s_dir}")

        if not label_dfs:
            raise RuntimeError("No labels.csv found! Check paths.")

        self.label_df = pd.concat(label_dfs, ignore_index=True)
        
        if 'image_name' not in self.label_df.columns:
             self.label_df['image_name'] = self.label_df.index

        print(f'Dataset Initialized. Loaded {len(self.label_df)} images from {len(self.shard_dirs)} shards.')

    def __len__(self):
        return len(self.label_df)

    def __getitem__(self, idx):
        img_path = os.path.join(
            self.label_df.loc[idx, 'shard_path'],
            'images',
            str(self.label_df.loc[idx, 'image_name'])
        )

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            # Return black image on error to prevent crash
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE))

        label = torch.tensor(self.label_df.loc[idx, 'label'], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

In [21]:
def evaluate(model, loader):
    model.eval()
    all_labels, all_probs = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images = images.to(DEVICE)
            logits = model(images).squeeze(1)
            probs = torch.sigmoid(logits)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    preds = (np.array(all_probs) >= 0.5).astype(int)
    acc = accuracy_score(all_labels, preds)
    auc = roc_auc_score(all_labels, all_probs)
    print(f"   >>> Val Accuracy: {acc:.4f} | AUC: {auc:.4f}")
    return acc

def train_model():
    # --- CONFIGURATION ---
    ACCUMULATION_STEPS = 4  # Simulate Batch Size 32 (8 * 4)
    
    # 1. Load Data
    full_dataset = AIGenDetDataset(TRAIN_SHARD_PATHS, transform=TRAIN_TRANSFORMS)
    val_size = int(len(full_dataset) * VAL_SPLIT)
    train_size = len(full_dataset) - val_size
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
    
    # IMPORTANT: Use drop_last=True to avoid incomplete batches messing up stats
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, 
                              num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, 
                            num_workers=2)

    # 2. Model Setup
    print("Loading EfficientNet-B4...")
    model = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
    model.to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR) # Keep 1e-4
    
    # Adjust scheduler for the REDUCED number of updates
    # Since we update only every 4 steps, total steps = len(loader) / 4
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=LR, 
        steps_per_epoch=len(train_loader) // ACCUMULATION_STEPS, 
        epochs=EPOCHS
    )

    print(f"Starting Training with Gradient Accumulation (Steps={ACCUMULATION_STEPS})...")
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad() # Initialize gradients
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for i, (images, labels) in enumerate(pbar):
            images = images.to(DEVICE)
            labels = labels.unsqueeze(1).to(DEVICE)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Normalize loss by accumulation steps
            loss = loss / ACCUMULATION_STEPS
            loss.backward()

            # Step ONLY every 'ACCUMULATION_STEPS'
            if (i + 1) % ACCUMULATION_STEPS == 0:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                
                # Scale loss back up for display
                current_loss = loss.item() * ACCUMULATION_STEPS
                running_loss += current_loss
                
                # Update progress bar less frequently to reduce jitter
                pbar.set_postfix({"Avg Loss": f"{running_loss / ((i+1)/ACCUMULATION_STEPS):.4f}"})

        # End of epoch validation
        val_acc = evaluate(model, val_loader)
        torch.save(model.state_dict(), f"effnet_b4_epoch_{epoch+1}.pth")

    return model

model = train_model()

Dataset Initialized. Loaded 100000 images from 2 shards.
Loading EfficientNet-B4...
Starting Training with Gradient Accumulation (Steps=4)...


Epoch 1/4:   0%|          | 0/11250 [00:00<?, ?it/s]

Validating:   0%|          | 0/1250 [00:00<?, ?it/s]

   >>> Val Accuracy: 0.9061 | AUC: 0.9703


Epoch 2/4:   0%|          | 0/11250 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/1250 [00:00<?, ?it/s]

   >>> Val Accuracy: 0.9331 | AUC: 0.9879


Epoch 4/4:   0%|          | 0/11250 [00:00<?, ?it/s]

Validating:   0%|          | 0/1250 [00:00<?, ?it/s]

   >>> Val Accuracy: 0.9374 | AUC: 0.9873


In [27]:
def run_inference(model, val_dir):
    print(f"Running Inference on: {val_dir}")
    model.eval()
    
    # Gather images
    val_path = Path(val_dir)
    image_files = sorted([f for f in val_path.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg']])
    
    if len(image_files) == 0:
        print("No images found! Check path.")
        return

    predictions = []
    with torch.no_grad():
        for img_file in tqdm(image_files, desc="Inferencing"):
            try:
                img = Image.open(img_file).convert("RGB")
                img_tensor = VAL_TRANSFORMS(img).unsqueeze(0).to(DEVICE)
                prob = torch.sigmoid(model(img_tensor)).item()
                
                predictions.append({"image_name": img_file.name, "score": prob})
            except Exception as e:
                print(f"Error {img_file.name}: {e}")

    df = pd.DataFrame(predictions)
    df.to_csv("submission.csv", index=False)
    print(f"Saved submission.csv with {len(df)} rows.")

# Run using the unlabeled path
run_inference(model, TEST_IMAGES_PATH)

Running Inference on: /kaggle/input/datasets/zeus005/validation-set-ivp/val_images


Inferencing:   0%|          | 0/10000 [00:00<?, ?it/s]

Saved submission.csv with 10000 rows.
